# Guardrail 3 — Retrieval

**Where it sits:** the retriever, before *and* after the vector search.

**What it stops:** query-rewrite attacks, namespace escape, semantic drift, un-cited claims sneaking into the prompt.

**Decision contract:** `{allow | rewrite | block, chunks[], dropped[], reasons[]}`

**Self-contained:** inlines a tiny RAG scaffold. No imports from other folders.

## Step 1 — toy RAG scaffold

In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

# Load .env from the same directory as this notebook (works in Jupyter)
_env_path = Path.cwd() / ".env"
if not _env_path.exists():
    _env_path = Path(__file__).parent / ".env" if "__file__" in globals() else _env_path
load_dotenv(_env_path, override=True)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage

# LangChain primitives, all driven by .env
LLM_MODEL     = os.getenv("MINIMAX_MODEL", "MiniMax-M3")
LLM_BASE_URL  = os.getenv("MINIMAX_BASE_URL", "https://api.minimax.io/v1")
LLM_API_KEY   = os.getenv("MINIMAX_API_KEY", "")

llm = ChatOpenAI(
    model=LLM_MODEL,
    api_key=LLM_API_KEY or "sk-fake",   # placeholder if no key -- calls will fail loudly
    base_url=LLM_BASE_URL,
    temperature=0,
)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=LLM_API_KEY or "sk-fake",
    base_url=LLM_BASE_URL,
)

print(f"LLM configured:  model={LLM_MODEL}  base_url={LLM_BASE_URL}")
print(f"API key loaded:  {'yes ('+LLM_API_KEY[:8]+'...)' if LLM_API_KEY else 'NO -- calls will fail; toy fallbacks below are unaffected'}")

# A safe wrapper so toy guardrail tests below stay deterministic.
# If FAKE_LLM=1 (or no key), use the toy. Otherwise call the real one.
_USE_FAKE = os.getenv("FAKE_LLM", "1") == "1" or not LLM_API_KEY

def chat(prompt: str, system: str = None) -> str:
    """Invoke the LangChain ChatOpenAI. Returns .content."""
    if _USE_FAKE:
        raise RuntimeError("chat() called but FAKE_LLM=1 -- use the toy LLM in this notebook's tests")
    msgs = []
    if system:
        msgs.append(SystemMessage(content=system))
    msgs.append(HumanMessage(content=prompt))
    return llm.invoke(msgs).content


In [ ]:
import re, math, hashlib

DOCS = [
    {"id": "d1", "text": "The capital of France is Paris.",   "namespace": "geo",     "source_uri": "kb://geo/fr"},
    {"id": "d2", "text": "The capital of Japan is Tokyo.",    "namespace": "geo",     "source_uri": "kb://geo/jp"},
    {"id": "d3", "text": "Q3 revenue was $4.2M, CFO J. Park.","namespace": "finance", "source_uri": "kb://finance/q3"},
    {"id": "d4", "text": "Today's lunch menu: pasta.",        "namespace": "misc",    "source_uri": "kb://misc/lunch"},
    {"id": "d5", "text": "Paris is also a person's name.",    "namespace": "geo",     "source_uri": "kb://geo/paris-name"},
]

def embed(text, dim=32):
    words = re.findall(r"[a-z0-9]+", text.lower())
    v = [0.0] * dim
    for w in words:
        h = int(hashlib.md5(w.encode()).hexdigest(), 16)
        v[h % dim] += 1.0
    n = math.sqrt(sum(x*x for x in v)) or 1.0
    return [x/n for x in v]

def cosine(a, b):
    return sum(x*y for x, y in zip(a, b))

def raw_retrieve(query, k=10):
    qv = embed(query)
    scored = [(cosine(qv, embed(d["text"])), d) for d in DOCS]
    scored.sort(reverse=True, key=lambda x: x[0])
    return scored[:k]

## Step 2 — retrieval guardrail

In [ ]:
def retrieval_guard(query: str,
                    allowed_namespaces=None,  # e.g. {"geo"} — None means all
                    k: int = 3,
                    sim_floor: float = 0.35,
                    overfetch_factor: int = 5):
    reasons, dropped = [], []

    # (a) BEFORE: namespace-escape attempt — strip any "namespace:X" the user tried to inject
    clean_query = re.sub(r"\bnamespace:\s*\S+", "", query, flags=re.I).strip()
    if clean_query != query:
        reasons.append("stripped_namespace_token")

    # (b) BEFORE: bound k — over-fetch then trim
    raw_k = min(max(k, 1) * overfetch_factor, len(DOCS))
    raw = raw_retrieve(clean_query, k=raw_k)

    kept = []
    for score, doc in raw:
        # (c) AFTER: namespace allow-list
        if allowed_namespaces is not None and doc["namespace"] not in allowed_namespaces:
            dropped.append({"id": doc["id"], "score": round(score,3), "reason": "namespace_disallowed"})
            continue

        # (d) AFTER: similarity floor
        if score < sim_floor:
            dropped.append({"id": doc["id"], "score": round(score,3), "reason": "below_sim_floor"})
            continue

        # (e) AFTER: dedup so one source can't dominate
        if any(c["id"] == doc["id"] for c in kept):
            dropped.append({"id": doc["id"], "score": round(score,3), "reason": "duplicate"})
            continue

        # (f) AFTER: citation hygiene — every chunk must carry provenance
        if "source_uri" not in doc or "id" not in doc:
            dropped.append({"id": doc.get("id","?"), "score": round(score,3), "reason": "missing_provenance"})
            continue

        kept.append({"id": doc["id"], "text": doc["text"], "source_uri": doc["source_uri"],
                     "namespace": doc["namespace"], "score": round(score,3)})

    # (g) trim to k
    kept = kept[:k]

    decision = "allow" if kept else "rewrite"
    if not kept:
        reasons.append("no_relevant_results")
    return {"decision": decision, "chunks": kept, "dropped": dropped, "reasons": reasons}

## Step 3 — test cases

In [ ]:
tests = [
    ("in-scope, geo",      "capital of France",           {"geo"},  3, 0.35),
    ("cross-namespace",    "Q3 revenue",                  {"geo"},  3, 0.35),
    ("namespace-inject",   "capital namespace:finance",   {"geo"},  3, 0.35),
    ("irrelevant query",   "what color is the sky",       None,     3, 0.35),
    ("high sim floor",     "capital of France",           None,     2, 0.95),
    ("k-cap respected",    "Paris Tokyo revenue lunch",   None,     2, 0.10),
]

for label, q, ns, k, sf in tests:
    r = retrieval_guard(q, allowed_namespaces=ns, k=k, sim_floor=sf)
    print(f"\n=== {label} ===")
    print(f"  decision: {r['decision']}   reasons: {r['reasons']}")
    print(f"  kept ({len(r['chunks'])}):   {[c['id'] for c in r['chunks']]}")
    print(f"  dropped ({len(r['dropped'])}): {[(d['id'], d['reason']) for d in r['dropped']]}")

## Step 4 — why citation hygiene matters

In [ ]:
# Imagine the index returns a chunk with no source_uri (corrupted ingest)
bad_chunk = {"id": "dX", "text": "This came from nowhere.", "namespace": "geo"}  # no source_uri

# The guard catches it:
import copy
DOCS_WITH_BAD = DOCS + [bad_chunk]
saved = raw_retrieve
def raw_retrieve(q, k=10):
    qv = embed(q)
    scored = [(cosine(qv, embed(d['text'])), d) for d in DOCS_WITH_BAD]
    scored.sort(reverse=True, key=lambda x: x[0])
    return scored[:k]
r = retrieval_guard("nonexistent query that matches nothing relevant", k=5, sim_floor=0.0)
print("kept ids:", [c['id'] for c in r['chunks']])
print("dropped reasons:", [d['reason'] for d in r['dropped']])

In [ ]:
### Real LangChain demo: FAISS-backed retriever with the guardrail

from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Build a small FAISS index from the toy corpus using real embeddings.
if not _USE_FAKE:
    lc_docs = [Document(page_content=d["text"], metadata=d) for d in DOCS]
    lc_vs = FAISS.from_documents(lc_docs, embeddings)
    lc_retriever = lc_vs.as_retriever(search_kwargs={"k": 10})
    raw = lc_retriever.invoke("capital of France")
    print(f"FAISS returned {len(raw)} raw docs")
else:
    raw = []
    print("[FAKE_LLM=1 -- real FAISS index requires the API.]")


## Takeaways

- **Namespace is a config knob, not a request parameter.** Never trust a user-supplied `collection:` or `namespace:` token — strip it and re-apply your allow-list server-side.
- **Over-fetch then filter.** Asking the vector store for `k=3` directly is brittle — you can't dedup or apply a similarity floor until you've seen the candidates. Over-fetch 5×, then trim.
- **Similarity floor is the only thing standing between you and a hallucinated answer to an irrelevant query.** Tune it on a labeled set; alert on regressions.
- **Citation hygiene at retrieval time, not at output time.** If a chunk lacks provenance, drop it at retrieval. The output guard (notebook 7) won't have anything to point at otherwise.

**Negative fixture checklist:** namespace escape, cross-namespace, irrelevant query, missing-provenance chunk. ✓